In [7]:
# Βιβλιοθήκες για τη διαχείριση αρχείων, δεδομένων και API requests
import os
import html
import requests
import pandas as pd

from dotenv import load_dotenv
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display


# Φόρτωση των μεταβλητών που είναι αποθηκευμένες στο αρχείο .env
load_dotenv()

# Ανάκτηση του YouTube API key χωρίς να εμφανίζεται μέσα στον κώδικα
API_KEY = os.getenv("YOUTUBE_API_KEY")

# Διακοπή της εκτέλεσης αν το API key δεν βρεθεί
if not API_KEY:
    raise ValueError("Δεν βρέθηκε το YOUTUBE_API_KEY στο αρχείο .env")

# Δημιουργία session για την επαναχρησιμοποίηση της σύνδεσης με το API
session = requests.Session()

print("Το API key φορτώθηκε επιτυχώς από το .env.")

Το API key φορτώθηκε επιτυχώς από το .env.


In [8]:
# Τα 4 θέματα και οι αντίστοιχοι όροι αναζήτησης στο YouTube
TOPICS = {
    "Football": "football",
    "Climate Change": "climate change",
    "Video Games": "video games",
    "Artificial Intelligence": "artificial intelligence"
}

# Αριθμός σχολίων που θέλουμε να συλλέξουμε για κάθε θέμα
TARGET_PER_TOPIC = 250

# Μέγιστος αριθμός βίντεο που θα αναζητηθούν για κάθε θέμα
VIDEOS_PER_TOPIC = 40

# Διευθύνσεις του YouTube API για αναζήτηση βίντεο και συλλογή σχολίων
SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
COMMENTS_URL = "https://www.googleapis.com/youtube/v3/commentThreads"

# Εμφάνιση μιας σύντομης περίληψης των ρυθμίσεων
print(f"Topics: {len(TOPICS)}")
print(f"Στόχος ανά topic: {TARGET_PER_TOPIC}")
print(f"Συνολικός στόχος: {TARGET_PER_TOPIC * len(TOPICS)} σχόλια")

Topics: 4
Στόχος ανά topic: 250
Συνολικός στόχος: 1000 σχόλια


In [9]:
# Λίστα στην οποία θα αποθηκευτούν τα στοιχεία των βίντεο
videos = []

# Αναζήτηση βίντεο ξεχωριστά για κάθε topic
for topic, search_query in TOPICS.items():

    # Παράμετροι που στέλνονται στο YouTube Search API
    params = {
        "part": "snippet",
        "q": search_query,
        "type": "video",
        "maxResults": VIDEOS_PER_TOPIC,
        "order": "relevance",
        "relevanceLanguage": "en",
        "safeSearch": "moderate",
        "key": API_KEY
    }

    # Αποστολή του αιτήματος στο API
    response = session.get(
        SEARCH_URL,
        params=params,
        timeout=30
    )

    # Διακοπή της διαδικασίας αν το API επιστρέψει σφάλμα
    if response.status_code != 200:
        error = response.json().get("error", {})

        raise RuntimeError(
            f"YouTube Search API error {response.status_code}: "
            f"{error.get('message', 'Unknown error')}"
        )

    # Μετατροπή της απάντησης του API σε Python dictionary
    data = response.json()

    # Καταγραφή των βασικών πληροφοριών κάθε βίντεο
    for search_rank, item in enumerate(
        data.get("items", []),
        start=1
    ):
        snippet = item["snippet"]
        video_id = item["id"]["videoId"]

        videos.append({
            "topic": topic,
            "search_query": search_query,
            "search_rank": search_rank,
            "video_id": video_id,
            "video_title": html.unescape(snippet["title"]),
            "video_channel": html.unescape(
                snippet["channelTitle"]
            ),
            "video_published_at": snippet["publishedAt"],
            "video_url": (
                f"https://www.youtube.com/watch?v={video_id}"
            )
        })


# Μετατροπή της λίστας σε DataFrame και αφαίρεση διπλότυπων βίντεο
videos_df = (
    pd.DataFrame(videos)
    .drop_duplicates(subset=["topic", "video_id"])
    .reset_index(drop=True)
)

# Έλεγχος του αριθμού των βίντεο που βρέθηκαν για κάθε topic
video_summary = (
    videos_df
    .groupby("topic")
    .agg(videos_found=("video_id", "nunique"))
    .reindex(TOPICS.keys())
)

display(video_summary)

print(f"Συνολικά videos: {len(videos_df)}")

,videos_found
topic,
Football,33
Climate Change,40
Video Games,40
Artificial Intelligence,40


Συνολικά videos: 153


In [10]:
def get_api_error(response):
    """Επιστρέφει τον λόγο και το μήνυμα ενός API error."""

    try:
        # Ανάκτηση των πληροφοριών του σφάλματος από την απάντηση
        error = response.json().get("error", {})
        errors = error.get("errors", [])

        reason = (
            errors[0].get("reason", "")
            if errors
            else ""
        )

        message = error.get(
            "message",
            "Unknown YouTube API error"
        )

    # Χρησιμοποιείται όταν η απάντηση δεν μπορεί να μετατραπεί σε JSON
    except ValueError:
        reason = ""
        message = response.text

    return reason, message


def fetch_comment_page(video_row, page_token=None):
    """Συλλέγει μία σελίδα από top-level σχόλια ενός βίντεο."""

    # Παράμετροι για τη συλλογή έως 100 σχολίων
    params = {
        "part": "snippet",
        "videoId": video_row["video_id"],
        "maxResults": 100,
        "order": "time",
        "textFormat": "plainText",
        "key": API_KEY
    }

    # Χρησιμοποιείται όταν ζητάμε την επόμενη σελίδα σχολίων
    if page_token:
        params["pageToken"] = page_token

    # Αποστολή αιτήματος στο YouTube Comments API
    response = session.get(
        COMMENTS_URL,
        params=params,
        timeout=30
    )

    # Διαχείριση πιθανών σφαλμάτων του API
    if response.status_code != 200:
        reason, message = get_api_error(response)

        # Περιπτώσεις στις οποίες το βίντεο παραλείπεται
        unavailable_reasons = {
            "commentsDisabled",
            "videoNotFound",
            "forbidden"
        }

        if reason in unavailable_reasons:
            print(
                f"Παράλειψη video {video_row['video_id']}: "
                f"{reason}"
            )
            return [], None

        # Τα υπόλοιπα σφάλματα σταματούν την εκτέλεση
        raise RuntimeError(
            f"YouTube Comments API error "
            f"{response.status_code}: {message}"
        )

    data = response.json()

    # Καταγραφή της ημερομηνίας και ώρας συλλογής
    collected_at = datetime.now(timezone.utc).isoformat()

    records = []

    # Επεξεργασία των σχολίων που επέστρεψε το API
    for item in data.get("items", []):
        thread_snippet = item["snippet"]
        top_comment = thread_snippet["topLevelComment"]
        comment_snippet = top_comment["snippet"]

        comment_id = top_comment["id"]

        # Το author channel ID μπορεί να μην υπάρχει σε όλα τα σχόλια
        author_channel = (
            comment_snippet
            .get("authorChannelId", {})
            .get("value")
        )

        # Προτιμάται το αρχικό κείμενο και χρησιμοποιείται εναλλακτικά το display text
        text = (
            comment_snippet.get("textOriginal")
            or comment_snippet.get("textDisplay", "")
        )

        # Αποθήκευση των διαθέσιμων στοιχείων του σχολίου
        records.append({
            "comment_id": comment_id,
            "author_name": html.unescape(
                comment_snippet.get(
                    "authorDisplayName",
                    ""
                )
            ),
            "author_channel_id": author_channel,
            "topic": video_row["topic"],
            "search_query": video_row["search_query"],
            "text": text,
            "published_at": comment_snippet.get(
                "publishedAt"
            ),
            "updated_at": comment_snippet.get(
                "updatedAt"
            ),
            "like_count": comment_snippet.get(
                "likeCount",
                0
            ),
            "reply_count": thread_snippet.get(
                "totalReplyCount",
                0
            ),
            "video_id": video_row["video_id"],
            "video_title": video_row["video_title"],
            "video_channel": video_row["video_channel"],
            "search_rank": video_row["search_rank"],
            "comment_url": (
                "https://www.youtube.com/watch?"
                f"v={video_row['video_id']}"
                f"&lc={comment_id}"
            ),
            "collected_at": collected_at
        })

    # Επιστρέφονται τα σχόλια και το token της επόμενης σελίδας
    return records, data.get("nextPageToken")

In [11]:
# Εδώ θα αποθηκευτεί το τελικό DataFrame κάθε topic
topic_datasets = []

# Χρησιμοποιείται για την αποφυγή διπλότυπων σχολίων μεταξύ των topics
global_comment_ids = set()

for topic in TOPICS:

    print(f"\nΣυλλογή topic: {topic}")

    # Επιλογή και ταξινόμηση των βίντεο του συγκεκριμένου topic
    topic_videos = (
        videos_df[videos_df["topic"] == topic]
        .sort_values("search_rank")
        .reset_index(drop=True)
    )

    # Προσωρινή αποθήκευση σχολίων και IDs για το τρέχον topic
    topic_pool = []
    topic_comment_ids = set()

    # Αποθήκευση του επόμενου page token για κάθε βίντεο
    next_page_tokens = {}

    # Συλλογή της πρώτης σελίδας σχολίων από κάθε βίντεο
    for _, video_row in topic_videos.iterrows():

        batch, next_token = fetch_comment_page(
            video_row
        )

        # Προσθήκη μόνο μοναδικών σχολίων
        for record in batch:
            comment_id = record["comment_id"]

            if (
                comment_id not in topic_comment_ids
                and comment_id not in global_comment_ids
            ):
                topic_pool.append(record)
                topic_comment_ids.add(comment_id)

        # Αποθήκευση του token αν υπάρχει επόμενη σελίδα
        if next_token:
            next_page_tokens[
                video_row["video_id"]
            ] = next_token

    # Συλλογή επιπλέον σελίδων μέχρι να φτάσουμε τα 250 σχόλια
    while (
        len(topic_pool) < TARGET_PER_TOPIC
        and next_page_tokens
    ):
        comments_before = len(topic_pool)

        for _, video_row in topic_videos.iterrows():
            video_id = video_row["video_id"]

            # Παράλειψη βίντεο που δεν έχει άλλη σελίδα σχολίων
            if video_id not in next_page_tokens:
                continue

            batch, next_token = fetch_comment_page(
                video_row,
                next_page_tokens[video_id]
            )

            # Προσθήκη μόνο σχολίων που δεν έχουν ήδη συλλεχθεί
            for record in batch:
                comment_id = record["comment_id"]

                if (
                    comment_id not in topic_comment_ids
                    and comment_id not in global_comment_ids
                ):
                    topic_pool.append(record)
                    topic_comment_ids.add(comment_id)

            # Ενημέρωση ή αφαίρεση του page token του βίντεο
            if next_token:
                next_page_tokens[video_id] = next_token
            else:
                del next_page_tokens[video_id]

            # Σταματάμε μόλις καλυφθεί ο στόχος
            if len(topic_pool) >= TARGET_PER_TOPIC:
                break

        # Προστασία από συνεχή επανάληψη αν δεν βρεθούν νέα σχόλια
        if len(topic_pool) == comments_before:
            break

    # Διακοπή αν δεν βρέθηκαν αρκετά σχόλια για το topic
    if len(topic_pool) < TARGET_PER_TOPIC:
        raise RuntimeError(
            f"Βρέθηκαν μόνο {len(topic_pool)} σχόλια "
            f"για το topic '{topic}'."
        )

    topic_df = pd.DataFrame(topic_pool)

    # Αρίθμηση των σχολίων μέσα σε κάθε βίντεο
    topic_df["position_in_video"] = (
        topic_df
        .groupby("video_id")
        .cumcount()
    )

    # Επιλογή 250 σχολίων με όσο γίνεται καλύτερη κατανομή μεταξύ των βίντεο
    topic_df = (
        topic_df
        .sort_values(
            [
                "position_in_video",
                "search_rank",
                "published_at"
            ],
            ascending=[True, True, False]
        )
        .head(TARGET_PER_TOPIC)
        .drop(columns="position_in_video")
        .reset_index(drop=True)
    )

    # Καταγραφή των τελικών comment IDs ώστε να μην επαναχρησιμοποιηθούν
    global_comment_ids.update(
        topic_df["comment_id"].tolist()
    )

    topic_datasets.append(topic_df)

    print(
        f"Ολοκληρώθηκε: {len(topic_df)} σχόλια "
        f"από {topic_df['video_id'].nunique()} videos"
    )


# Ένωση των τεσσάρων topics σε ένα ενιαίο DataFrame
comments_df = pd.concat(
    topic_datasets,
    ignore_index=True
)


Συλλογή topic: Football
Ολοκληρώθηκε: 250 σχόλια από 31 videos

Συλλογή topic: Climate Change
Παράλειψη video aasX7koCrQ8: commentsDisabled
Παράλειψη video SDRxfuEvqGg: commentsDisabled
Παράλειψη video k3yL_1L85Mk: commentsDisabled
Παράλειψη video ARB6R51iQ_k: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 35 videos

Συλλογή topic: Video Games
Παράλειψη video mDWJipt_rN4: commentsDisabled
Παράλειψη video Uen-aD8kTMY: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 38 videos

Συλλογή topic: Artificial Intelligence
Παράλειψη video _19pRsZRiz4: commentsDisabled
Παράλειψη video JcXKbUIebrU: commentsDisabled
Παράλειψη video qD-o2lDQwa8: commentsDisabled
Παράλειψη video ttIOdAdQaUE: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 36 videos


In [13]:
# Τελικές στήλες και σειρά εμφάνισής τους στο dataset
FINAL_COLUMNS = [
    "comment_id",
    "author_name",
    "author_channel_id",
    "topic",
    "search_query",
    "text",
    "published_at",
    "updated_at",
    "like_count",
    "reply_count",
    "video_id",
    "video_title",
    "video_channel",
    "search_rank",
    "comment_url",
    "collected_at"
]

comments_df = comments_df[FINAL_COLUMNS]

# Περίληψη των σχολίων και των βίντεο ανά topic
summary = (
    comments_df
    .groupby("topic")
    .agg(
        comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        videos_used=("video_id", "nunique")
    )
    .reindex(TOPICS.keys())
)

display(summary)

# Έλεγχος για διπλότυπα comment IDs
duplicate_ids = comments_df["comment_id"].duplicated().sum()

print(f"Συνολικά σχόλια: {len(comments_df)}")
print(f"Διπλότυπα comment IDs: {duplicate_ids}")

# Αυτόματοι έλεγχοι για την επιβεβαίωση του τελικού dataset
assert len(comments_df) == 1000
assert comments_df["comment_id"].is_unique
assert (
    comments_df.groupby("topic")
    .size()
    .eq(TARGET_PER_TOPIC)
    .all()
)

# Δημιουργία του φακέλου data/raw 
output_folder = Path("../data/raw")
output_folder.mkdir(parents=True, exist_ok=True)

# Όνομα και διαδρομή του τελικού raw αρχείου
output_file = output_folder / "youtube_comments_raw.csv"

# Αποθήκευση σε CSV 
comments_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nΑποθηκεύτηκε επιτυχώς στο: {output_file}")

,comments,unique_comments,videos_used
topic,,,
Football,250,250,31
Climate Change,250,250,35
Video Games,250,250,38
Artificial Intelligence,250,250,36


Συνολικά σχόλια: 1000
Διπλότυπα comment IDs: 0

Αποθηκεύτηκε επιτυχώς στο: ../data/raw/youtube_comments_raw.csv


In [16]:
# Εμφάνιση 20 ενδεικτικών σχολίων από κάθε topic

SAMPLE_PER_TOPIC = 20

SAMPLE_COLUMNS = [
    "comment_id",
    "author_name",
    "author_channel_id",
    "topic",
    "search_query",
    "text",
    "published_at",
    "updated_at",
    "like_count",
    "reply_count",
    "video_id",
    "video_title",
    "video_channel",
    "search_rank",
    "comment_url",
    "collected_at"
]

for topic in TOPICS:

    topic_sample = (
        comments_df[
            comments_df["topic"] == topic
        ][SAMPLE_COLUMNS]
        .head(SAMPLE_PER_TOPIC)
        .reset_index(drop=True)
    )

    print(f"\n{topic}: {len(topic_sample)} ενδεικτικά σχόλια")

    with pd.option_context(
        "display.max_colwidth",
        200
    ):
        display(topic_sample)


Football: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,Ugyhj05JeMNyWipqxVp4AaABAg,@tokkyuuressha,UCu60UWqGRo_6NF6PDf-O1oQ,Football,football,8:34 what a clown piece from Ramos,2026-08-06T21:49:28Z,2026-08-06T21:49:28Z,0,0,moRX1XoBUFo,"Ronaldo, Messi, Neymar, Mbappe Shocked the World in Their Final Match",ArtSoccer,1,https://www.youtube.com/watch?v=moRX1XoBUFo&lc=Ugyhj05JeMNyWipqxVp4AaABAg,2026-08-07T09:46:01.721712+00:00
1,UgyhV_Isqm3AhYD0IOx4AaABAg,@따뜻한밥-t7q,UC7rljqa6-4WJPRuaeR9ho9Q,Football,football,저 ㅈㄹ을 해도 막판에 골을 못 넣는데...,2026-08-06T13:07:25Z,2026-08-06T13:07:25Z,0,0,02h7MB4Re4o,Best Football Skills 2026,SportsHD,2,https://www.youtube.com/watch?v=02h7MB4Re4o&lc=UgyhV_Isqm3AhYD0IOx4AaABAg,2026-08-07T09:46:01.888698+00:00
2,UgxLBvsU6eZhWhcYA1x4AaABAg,@TheWingroveFamily,UCyY4jVInowKG9Nm20xXGHFw,Football,football,🚨YES GUYS! Who's team is better? Team Billy & Dustie or Team Roman & Amelie? Let us know!!⬇,2026-07-10T15:08:13Z,2026-07-10T15:08:13Z,120,72,qHIiLIemo6k,INSANE WORLD CUP FC26 CARD BATTLE!!,The Wingrove Family,3,https://www.youtube.com/watch?v=qHIiLIemo6k&lc=UgxLBvsU6eZhWhcYA1x4AaABAg,2026-08-07T09:46:02.075941+00:00
3,UgzYV_zK1zww1WW0Z5p4AaABAg,@Sathyabhama-z3d,UCkRfe1pHRCU3Weq3SJV40vQ,Football,football,Vbvgdff,2026-08-07T04:10:05Z,2026-08-07T04:10:05Z,0,1,pcAPOSv0OjA,SPAIN 1-0 ARGENTINA | FIFA WORLD CUP 2026 FINAL | FULL LIVE,BOBOLA TV,4,https://www.youtube.com/watch?v=pcAPOSv0OjA&lc=UgzYV_zK1zww1WW0Z5p4AaABAg,2026-08-07T09:46:02.297134+00:00
4,Ugx-muWaVm7bSd_DNs94AaABAg,@AyomideOladipo-y2h,UCFVMGBH_VYmw_-LbpcLCvpw,Football,football,Volley looks easy but when you play football you know it's not,2026-08-07T09:02:13Z,2026-08-07T09:02:13Z,0,0,ICQiBryLdMY,Most Inside Inside Foot Volleys In Football #football #shorts #volley,OG_Clips,5,https://www.youtube.com/watch?v=ICQiBryLdMY&lc=Ugx-muWaVm7bSd_DNs94AaABAg,2026-08-07T09:46:02.412103+00:00
5,UgzemOqfxcVo9huSNxZ4AaABAg,@thebluehorizon_ai,UCpIWVf6GQzlrQHYVHPryGaA,Football,football,Listen to the song in Spotify https://open.spotify.com/album/57GJ7lFpk9qXd2u2np7gMn,2026-06-25T04:01:35Z,2026-06-25T04:01:35Z,310,31,7RogQiih3M8,WORLD CUP 2026 🏆 The AI Football Movie,The Blue Horizon AI,6,https://www.youtube.com/watch?v=7RogQiih3M8&lc=UgzemOqfxcVo9huSNxZ4AaABAg,2026-08-07T09:46:02.631667+00:00
6,UgxdAiHE7IcsKZan0xB4AaABAg,@DavidCebotari-x1f,UCrzC7EsAoH9WVF5d0wc3Ykg,Football,football,Офсаид😂,2026-08-07T09:42:06Z,2026-08-07T09:42:06Z,0,0,v15gzK3ljXg,Самое трогательное прощание #football #футбол #respect #worldcup #ronaldo #messi #mbappe #sad,Kurofoot,7,https://www.youtube.com/watch?v=v15gzK3ljXg&lc=UgxdAiHE7IcsKZan0xB4AaABAg,2026-08-07T09:46:02.762743+00:00
7,Ugx-gpUUICJrswsELOl4AaABAg,@GridironGoals-q4b,UCE6C7mqqD1piAVUH0barzxg,Football,football,"Thanks for adding those cartoon sounds, otherwise i wouldn't know how to feel",2026-08-05T09:26:22Z,2026-08-05T09:26:22Z,0,0,aUL3haqhg_8,45 Minutes of The Most Cinematic Moments in Football History,FOOT DAILY,8,https://www.youtube.com/watch?v=aUL3haqhg_8&lc=Ugx-gpUUICJrswsELOl4AaABAg,2026-08-07T09:46:02.952554+00:00
8,UgylJSUbPOVMW3Io6Ux4AaABAg,@ClipboardSports,UCPupL-FeKFXpHW-QFAfsJzw,Football,football,Shake then red card,2026-08-07T08:07:51Z,2026-08-07T08:07:51Z,0,0,1RgFCz9qdhk,Controversial Red Cards In Football 🟥🤯,Galinho FC,9,https://www.youtube.com/watch?v=1RgFCz9qdhk&lc=UgylJSUbPOVMW3Io6Ux4AaABAg,2026-08-07T09:46:03.171334+00:00
9,UgyR2PQD_ZodUbZ-4l54AaABAg,@NinetyVirus,UC9aSRBH_gEtPD5E7cWO10Dw,Football,football,"Greatest game on earth, but also the scariest? 🥶",2026-05-08T10:35:10Z,2026-05-08T10:35:10Z,113,12,9VnSsqGdhZA,Scariest Moments In Football,Ninety,10,https://www.youtube.com/watch?v=9VnSsqGdhZA&lc=UgyR2PQD_ZodUbZ-4l54AaABAg,2026-08-07T09:46:03.369495+00:00



Climate Change: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgxudeMdc2qaoAvm3hd4AaABAg,@protonflux5766,UCmGUQ83i9FwU7yS6yaGJ0lg,Climate Change,climate change,"The climate is by definition a dynamic, evolving system. It will continue to change for the better or worse whether we want it to or not.",2026-08-05T05:28:23Z,2026-08-05T05:28:23Z,0,1,xBGSlxv38nI,2026 isn't breaking heat records - yet. El Nino is forecast to spike temperatures,Associated Press,1,https://www.youtube.com/watch?v=xBGSlxv38nI&lc=UgxudeMdc2qaoAvm3hd4AaABAg,2026-08-07T09:46:09.071333+00:00
1,Ugx1flrOhhmUqQSEv5p4AaABAg,@Hammy712,UCfNSYWX44G0CUDG8mrxO8cg,Climate Change,climate change,Wow look what happens when you put engineers and not businessmen and con artists in charge.,2026-08-07T00:23:21Z,2026-08-07T00:23:21Z,3,0,bUyVQun964Y,"""It's clear to me that climate change is the single greatest threat to human security that we face""",Green Party of England & Wales,2,https://www.youtube.com/watch?v=bUyVQun964Y&lc=Ugx1flrOhhmUqQSEv5p4AaABAg,2026-08-07T09:46:09.225637+00:00
2,UgyvYk15M60ABhEJkjp4AaABAg,@chrishughes7627,UCLKMsPxDYpLnBoG1aAxkVCg,Climate Change,climate change,"Laura,tell the real reason our temperatures are increasing. Clear skies and lack of pollution, nothing to reflect the sun back into space,hence the sun is now like a super powered led light which ...",2026-07-24T06:35:36Z,2026-07-24T06:35:36Z,0,0,nwzIpxPsy2E,Record heatwave 'impossible' without climate change #weather #heatwave #climatechange,ITV News,4,https://www.youtube.com/watch?v=nwzIpxPsy2E&lc=UgyvYk15M60ABhEJkjp4AaABAg,2026-08-07T09:46:09.463462+00:00
3,UgwAYiw1izwrlDac8vF4AaABAg,@moxiesaint-clare4257,UCb1b1kpmrN10bAMGuf8z1GQ,Climate Change,climate change,"I remember building a snowman in the back garden 1978-1985 , stayed there for weeks. We had snow every winter but it got less and less each year. Summers were warm 18-23 c wasn't until 97, 98 the...",2026-07-11T04:01:06Z,2026-07-11T04:01:06Z,0,0,DOPcMMZ6FHU,1976 was an outlier. This isn't. ☀️ #summer #sun #climatechange #physics #nature,The Royal Society,5,https://www.youtube.com/watch?v=DOPcMMZ6FHU&lc=UgwAYiw1izwrlDac8vF4AaABAg,2026-08-07T09:46:09.572711+00:00
4,Ugz92yE2b4esEZ5h_354AaABAg,@lkenaa9038,UCxdAZzqHbgEuEy6MMzKUYfw,Climate Change,climate change,"The climate also changed in Joseph's time, there were 7 good years followed by 7 bad years.",2026-07-14T07:27:15Z,2026-07-14T07:27:15Z,0,0,wmYcLgSLpUQ,Jeremy Clarkson says farming has given him a new perspective on the changing climate.,Times News,6,https://www.youtube.com/watch?v=wmYcLgSLpUQ&lc=Ugz92yE2b4esEZ5h_354AaABAg,2026-08-07T09:46:09.754868+00:00
5,UgxgZCzSDlAXrU_V2qB4AaABAg,@PEdulis,UC6vplwkxFhHTCLMPWYPmFsA,Climate Change,climate change,You think man made climate change is wild?\nWait until climate changes humans!,2026-08-06T21:41:55Z,2026-08-06T21:41:55Z,0,0,LhhpDkThn-o,Climate crisis is 'biggest threat since WW2': Is this hysterics or hard truth?,LBC,7,https://www.youtube.com/watch?v=LhhpDkThn-o&lc=UgxgZCzSDlAXrU_V2qB4AaABAg,2026-08-07T09:46:09.920909+00:00
6,Ugw7GQnitlXXaJXTghx4AaABAg,@AddiK-k5d,UCv91zphNFBjkDVl_rIIsqgQ,Climate Change,climate change,This is bad for the environment,2026-07-25T23:23:11Z,2026-07-25T23:23:11Z,0,0,OGkTjqsqU5c,"Global sea levels could rise more than expected because of climate change, according to a new study",NPR,8,https://www.youtube.com/watch?v=OGkTjqsqU5c&lc=Ugw7GQnitlXXaJXTghx4AaABAg,2026-08-07T09:46:10.100541+00:00
7,Ugwd3VY0pb9Fjej6OTV4AaABAg,@jakobusphsteyn3500,UCOel3cqGtG8WqmxjY2z3_TQ,Climate Change,climate change,Climate change has always been on the charts and will always be. The reality of the earth's cyclical exitance climate wise is written in the geological history of the planet and there is nothing h...,2026-08-07T07:12:27Z,2026-08-07T07:12:27Z,0,0,vy-P6C2F0Wg,JUST IN: Sheldon Whitehouse And Ron Johnson G


Video Games: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgznY-monTOvBejwRb14AaABAg,@JalwaSherAfgan,UCudxnnkiHl-sr04ZpqcisAA,Video Games,video games,If I'm alive I'll come again ..... How nostalgic it is .....,2026-08-07T09:30:17Z,2026-08-07T09:30:17Z,0,0,cE6wxDqdOV0,Lana Del Rey - Video Games,LanaDelReyVEVO,1,https://www.youtube.com/watch?v=cE6wxDqdOV0&lc=UgznY-monTOvBejwRb14AaABAg,2026-08-07T09:46:16.786047+00:00
1,UgxNzCD7pmF1o7AWv9F4AaABAg,@LostPanda,UCyh5g11KbG_YdbRw1ktAJqA,Video Games,video games,*Do you play video games?*,2024-03-27T12:52:55Z,2024-03-27T12:52:55Z,1032,71,hAV9ideCFEc,Lana Del Rey - Video Games (Lyrics),Lost Panda,2,https://www.youtube.com/watch?v=hAV9ideCFEc&lc=UgxNzCD7pmF1o7AWv9F4AaABAg,2026-08-07T09:46:16.997071+00:00
2,UgwhRIIu7RuhHu1JBM94AaABAg,@Astitwa0,UCwt18qvdcGbs-BLYQeIVpOQ,Video Games,video games,Then GPU and Then AI too,2026-08-07T09:35:29Z,2026-08-07T09:35:29Z,0,0,bRsbf5h5qmw,How Video Games Evolved in 68 Years 🎮 #shorts,Learn Tech Gaming,3,https://www.youtube.com/watch?v=bRsbf5h5qmw&lc=UgwhRIIu7RuhHu1JBM94AaABAg,2026-08-07T09:46:17.200676+00:00
3,UgyYWIN67DM8KePeWtt4AaABAg,@shaneallen231,UCGllR9l6O4RD3F4eCzY9PSw,Video Games,video games,I want you to still play fucking video games.,2026-08-07T05:58:45Z,2026-08-07T05:58:55Z,0,0,GNJtPFXUnm4,Tenacious D - Video Games,Tenacious D,4,https://www.youtube.com/watch?v=GNJtPFXUnm4&lc=UgyYWIN67DM8KePeWtt4AaABAg,2026-08-07T09:46:17.450948+00:00
4,UgwK_L00eJC4DZkik5h4AaABAg,@Mirai-u2i,UCXkRha3Nho6Qnr6jBdRCiwg,Video Games,video games,❤😂😂😂😂😂😂😂😂😂😂,2026-08-06T21:54:57Z,2026-08-06T21:54:57Z,0,0,wnkLpGhJh-M,Cool creative smart fun game videos that you can't miss. #100,Great_Painter,5,https://www.youtube.com/watch?v=wnkLpGhJh-M&lc=UgwK_L00eJC4DZkik5h4AaABAg,2026-08-07T09:46:17.676288+00:00
5,Ugz2g2P8PkL0N2pn_XR4AaABAg,@samanthasalazar486,UCHb-oaFErH2KzUfGqqec59g,Video Games,video games,Request please upload now with enough time all characters losing animations in London party mode rhythmic ribbon with knuckles in all songs synchronized swimming all songs relay and dream rafting ...,2026-06-02T02:05:04Z,2026-06-02T02:05:04Z,3,0,jw5M9ByzVRo,Mario & Sonic At The London 2012 Olympic Games Football Team Sonic Gameplay,Minh Party U,6,https://www.youtube.com/watch?v=jw5M9ByzVRo&lc=Ugz2g2P8PkL0N2pn_XR4AaABAg,2026-08-07T09:46:17.783304+00:00
6,UgzKOvCeYlsoSePjRzV4AaABAg,@PlayverseGaming-z6c,UCf9t1PO6S051hoqGvBgfMNg,Video Games,video games,00:26 - God of War 2018\r\n01:34 - Hollow Knight\r\n02:37 - Days Gone\r\n03:56 - Resident Evil 7\r\n05:01 - Clair Obscur: Expedition 33\r\n06:22 - A Plague Tale Innocence\r\n07:50 - The Last of Us...,2026-08-03T12:45:42Z,2026-08-03T12:45:42Z,6,0,n7brfo8-mmg,16 PERFECT 10/10 Games That Will Make You Love Gaming Again!,Playverse Gaming,7,https://www.youtube.com/watch?v=n7brfo8-mmg&lc=UgzKOvCeYlsoSePjRzV4AaABAg,2026-08-07T09:46:17.888660+00:00
7,Ugw8IDn1gQwmDu9uZTV4AaABAg,@Kittistory,UC7jUluhVVagVK-JQfEU7s1w,Video Games,video games,The spongebob squarepant super sponge anti piocy screen is not e.D it's just the wheel laying that tries to choose,2026-07-29T04:31:16Z,2026-07-29T04:31:16Z,1,0,gM3lOz0Gblk,How Video Games Punish Stealers,Sambucha,8,https://www.youtube.com/watch?v=gM3lOz0Gblk&lc=Ugw8IDn1gQwmDu9uZTV4AaABAg,2026-08-07T09:46:18.208543+00:00
8,UgwDxwPWw75AdL7b4gV4AaABAg,@Bitcenteplay,UCTm0ZgrmfFAbxQ3idLPrNkA,Video Games,video games,"""Empecé jugando con mi hijo y ahora me gana más seguido de lo que me gustaría. 😂""",2026-08-05T22:39:51Z,2026-08-05T22:39:51Z,0,0,J0xw8S5PpCg,Mario Kart 8 Deluxe - All New DLC Courses (DLC Booster Pack 1),Sirloin,9,https://www.youtube.com/watch?v=J0xw8S5PpCg&lc=UgwDxwPWw75AdL7b4gV4AaABAg,2026-08-07T09:46:18.409383+00:00
9,UgxmbVGCwSabv_vuDvR4AaABAg,@Hbbdh22eh,UCog8s3B0NhCIxS5PhEZiPFA,Video Games,video games,Touch some grass,2026-08-06T12:35:21Z,2026-08-06T12:35:21Z,0,0,A0NfOm


Artificial Intelligence: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgwNlF1Du2raVtCnNxB4AaABAg,@FutureBusinessTech,UCGBO6EahCqQSyXIws1MQdDg,Artificial Intelligence,artificial intelligence,Feel free to like and subscribe if you enjoyed the video. Watch this next video about the Technological Singularity: https://youtu.be/yHEnKwSUzAE.,2023-10-28T15:56:47Z,2023-10-28T15:56:47Z,129,26,tFx_UNW9I1U,The 10 Stages of Artificial Intelligence,Future Business Tech,2,https://www.youtube.com/watch?v=tFx_UNW9I1U&lc=UgwNlF1Du2raVtCnNxB4AaABAg,2026-08-07T09:46:25.875229+00:00
1,Ugw5IyCxOlVoGqbBZkd4AaABAg,@SimplilearnOfficial,UCsvqVGtbbyHaMoevxPAq9Fg,Artificial Intelligence,artificial intelligence,"""🔥Michigan Engineering Professional Certificate in AI and Machine Learning - https://www.simplilearn.com/professional-aiml-program?utm_campaign=uMzUB89uSxU&utm_medium=Comments&utm_source=Youtube\n...",2023-05-24T12:25:28Z,2026-07-29T05:50:02Z,18,2,uMzUB89uSxU,What is Artificial Intelligence? | Artificial Intelligence In 5 Minutes | AI Explained | Simplilearn,Simplilearn,3,https://www.youtube.com/watch?v=uMzUB89uSxU&lc=Ugw5IyCxOlVoGqbBZkd4AaABAg,2026-08-07T09:46:26.094670+00:00
2,UgwH_CFvCOPxiasprSF4AaABAg,@yannisth900,UCZu3sHqilfCX90Z8YuJ48tA,Artificial Intelligence,artificial intelligence,"The argument that ""AI can't generate new content"" when it comes to forms of art is based on the fact that genAI is trained on existing content, and since the AI doesn't have a soul / can't have or...",2026-08-05T11:40:47Z,2026-08-05T11:40:47Z,0,0,qYNweeDHiyU,"AI, Machine Learning, Deep Learning and Generative AI Explained",IBM Technology,4,https://www.youtube.com/watch?v=qYNweeDHiyU&lc=UgwH_CFvCOPxiasprSF4AaABAg,2026-08-07T09:46:26.348807+00:00
3,UgxcO-13M-e7E7mwi3Z4AaABAg,@JoelMiller-0,UCQC9z70s3zSt7DRtXWQR9sA,Artificial Intelligence,artificial intelligence,Insert SR-72 Edit*,2026-08-06T21:32:43Z,2026-08-06T21:32:43Z,0,0,F-pMp8AaXvw,artificial intelligence(SLOWED),JG,5,https://www.youtube.com/watch?v=F-pMp8AaXvw&lc=UgxcO-13M-e7E7mwi3Z4AaABAg,2026-08-07T09:46:26.569229+00:00
4,Ugzcr5Ae4IkGyRno4_t4AaABAg,@LadifewaxNamitey,UCQvDUeHUZConuYCMyYk8sbQ,Artificial Intelligence,artificial intelligence,"Instant subscriber, your content is outstanding!",2025-12-05T21:48:33Z,2025-12-05T21:48:33Z,74,4,m8o2GrbR3d8,SIMPLEST Explanation of How Artificial Intelligence Works? No Jargon | What is AI? How AI works?,Science Simplified 4 All,6,https://www.youtube.com/watch?v=m8o2GrbR3d8&lc=Ugzcr5Ae4IkGyRno4_t4AaABAg,2026-08-07T09:46:26.799836+00:00
5,Ugz9M1uAultfD2bo4EB4AaABAg,@reaazali6347,UCE07cVjro8vIoGZ3_B2F5Dw,Artificial Intelligence,artificial intelligence,Well they should have stopped the ahole who invented this AI shit in the first place,2026-08-07T09:05:51Z,2026-08-07T09:05:51Z,0,0,siHhK75gf60,Godfather Of AI: We Should Prepare For What's Coming In 2030,Neural Nutshell,7,https://www.youtube.com/watch?v=siHhK75gf60&lc=Ugz9M1uAultfD2bo4EB4AaABAg,2026-08-07T09:46:27.052521+00:00
6,UgwRrws1r2evYsc9zV14AaABAg,@SimplilearnOfficial,UCsvqVGtbbyHaMoevxPAq9Fg,Artificial Intelligence,artificial intelligence,"""🔥Michigan Engineering Professional Certificate in AI and Machine Learning - https://www.simplilearn.com/professional-aiml-program?utm_campaign=ad79nYk2keg&utm_medium=Comments&utm_source=Youtube\n...",2021-09-08T12:54:19Z,2026-07-29T05:45:40Z,214,8,ad79nYk2keg,What Is AI? | Artificial Intelligence | What is Artificial Intelligence? | AI In 5 Mins |Simplilearn,Simplilearn,8,https://www.youtube.com/watch?v=ad79nYk2keg&lc=UgwRrws1r2evYsc9zV14AaABAg,2026-08-07T09:46:27.292374+00:00
7,UgwSWfaGoEMh_YYLZSl4AaABAg,@JeffSu,UCwAnu01qlnVg1Ai2AbtTMaA,Artificial Intelligence,artificial intelligence,Who else thought ChatGPT and AI were the same thing? 🙋🏻\n\nTIMESTAMPS\n00:00 Google’s AI Course in 10 Minutes\n00:38 What is Artificial Intelligence?\n01:27 What is Machine Learni